In [0]:
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window
import mlflow
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from datetime import datetime

CATALOG     = "p2p_databricks"
VS_ENDPOINT = "p2p"
PO_INDEX    = f"{CATALOG}.silver_vs.po_merged_index"
GR_INDEX    = f"{CATALOG}.silver_vs.gr_merged_index"
INV_INDEX   = f"{CATALOG}.silver_vs.inv_merged_index"

GT_PATH     = f"/Volumes/{CATALOG}/staging/p2p_files/ground_truth.csv"

vsc = VectorSearchClient(disable_notice=True)
print("✅ Config ready")

In [0]:
# Load existing Gold results — this is Method A
gold_current = spark.table(f"{CATALOG}.gold.reconciliation_results")

gt = spark.read.csv(GT_PATH, header=True, inferSchema=True)

method_a = (
    gt.join(
        gold_current.select(
            "invoice_number",
            F.col("reason_code").alias("method_a_reason"),
            F.col("match_status").alias("method_a_status"),
            F.col("vendor_similarity_score").alias("method_a_vendor_sim"),
            F.col("amount_deviation_pct").alias("method_a_deviation")
        ),
        on="invoice_number", how="left"
    )
    .withColumn("expected_status",
        F.when(F.col("expected_overall_match") == True,
               F.lit("APPROVED")).otherwise(F.lit("EXCEPTION"))
    )
    .withColumn("method_a_correct",
        F.col("method_a_status") == F.col("expected_status")
    )
)

a_total   = method_a.count()
a_correct = method_a.filter("method_a_correct = true").count()
a_acc     = round(a_correct / a_total * 100, 1)

print(f"Method A — Exact join + SequenceMatcher")
print(f"  Accuracy: {a_correct}/{a_total} = {a_acc}%")

In [0]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

po_index_name  = PO_INDEX
vs_endpoint    = VS_ENDPOINT

# Schema for VS results
vs_result_schema = T.ArrayType(T.StructType([
    T.StructField("po_number",   T.StringType()),
    T.StructField("vendor_name", T.StringType()),
    T.StructField("vs_score",    T.FloatType()),
]))

@pandas_udf(vs_result_schema)
def po_vector_search_udf(descriptions: pd.Series) -> pd.Series:
    """
    Query PO Vector Search index using invoice merged_description.
    Returns top 3 candidate POs with similarity scores.
    """
    client = VectorSearchClient(disable_notice=True)
    index  = client.get_index(vs_endpoint, po_index_name)

    def search(text):
        if not text:
            return []
        try:
            results = index.similarity_search(
                query_text  = str(text),
                columns     = ["po_number", "vendor_name"],
                num_results = 3
            )
            out = []
            for r in results.get("result", {}).get("data_array", []):
                out.append({
                    "po_number":   str(r[0]) if r[0] else None,
                    "vendor_name": str(r[1]) if r[1] else None,
                    "vs_score":    float(r[2]) if r[2] else 0.0,
                })
            return out
        except Exception as e:
            return []

    return descriptions.apply(search)

print("✅ VS UDF defined")

In [0]:
inv_silver = spark.table(f"{CATALOG}.silver_vs.invoice")
po_silver  = spark.table(f"{CATALOG}.silver.purchase_order")
gr_silver  = spark.table(f"{CATALOG}.silver.good_receipt")

# Step 1: Run VS on all invoices
print("Running Vector Search on all invoices...")
inv_with_vs = inv_silver.withColumn(
    "vs_po_candidates",
    po_vector_search_udf(F.col("merged_description"))
)

# Step 2: Extract best VS candidate
inv_with_vs = (
    inv_with_vs
    .withColumn("vs_best_po",
                F.col("vs_po_candidates")[0]["po_number"])
    .withColumn("vs_best_score",
                F.col("vs_po_candidates")[0]["vs_score"])
    .withColumn("vs_best_vendor",
                F.col("vs_po_candidates")[0]["vendor_name"])
    # Rename invoice po_number before join to avoid ambiguity
    .withColumnRenamed("po_number", "inv_po_number")
)

# Step 3: Join Invoice → PO using renamed column
# FIX: use inv_po_number (renamed) to avoid ambiguous po_number reference
inv_po = (
    inv_with_vs.join(
        po_silver.select(
            F.col("po_number").alias("po_po_number"),
            F.col("vendor_name").alias("po_vendor_name"),
            F.col("total_amount_aed").alias("po_total_amount"),
        ),
        on=F.col("inv_po_number") == F.col("po_po_number"),
        how="left"
    )
    .withColumn("resolved_po_number", F.col("inv_po_number"))
    .withColumn("po_found", F.col("po_vendor_name").isNotNull())
    .drop("po_po_number")
)

# Step 4: Join → GR
inv_po_gr = (
    inv_po.join(
        gr_silver.select(
            F.col("po_number").alias("gr_po_number"),
            "gr_number", "gr_date", "gr_status",
            F.col("line_items").alias("gr_line_items"),
        ),
        on=F.col("resolved_po_number") == F.col("gr_po_number"),
        how="left"
    )
    .withColumn("gr_found", F.col("gr_number").isNotNull())
    .drop("gr_po_number")
)

print(f"Invoices with VS candidates : {inv_with_vs.count()}")
print(f"After PO join — matched     : {inv_po.filter('po_found').count()}")
print(f"After GR join — matched     : {inv_po_gr.filter('gr_found').count()}")

In [0]:
@F.udf(returnType=T.DoubleType())
def vs_vendor_similarity(inv_vendor: str, vs_vendor: str) -> float:
    if not inv_vendor or not vs_vendor:
        return None
    from difflib import SequenceMatcher
    import re
    def norm(s):
        s = s.lower().strip()
        s = re.sub(r'\b(llc|fz llc|ltd|limited|fz)\b', '', s)
        return re.sub(r'\s+', ' ', s).strip()
    return round(
        SequenceMatcher(None, norm(inv_vendor), norm(vs_vendor)).ratio(), 4
    )

# Use explicit column reference for invoice vendor_name
# After the join, invoice vendor_name is still unambiguous
# (PO vendor was renamed to po_vendor_name)
inv_po_gr = inv_po_gr.withColumn(
    "vs_vendor_sim",
    vs_vendor_similarity(
        F.col("vendor_name"),      # invoice vendor (unambiguous after rename)
        F.col("vs_best_vendor")    # VS top result vendor
    )
)

# Duplicate detection — use resolved_po_number (unambiguous)
dup_window   = Window.partitionBy("resolved_po_number").orderBy("invoice_number")
count_window = Window.partitionBy("resolved_po_number")

inv_po_gr = (
    inv_po_gr
    .withColumn("_rank",  F.row_number().over(dup_window))
    .withColumn("_count", F.count("invoice_number").over(count_window))
    .withColumn("is_duplicate",
                (F.col("_count") > 1) & (F.col("_inv_rank") > 1)
                if False else  # placeholder
                (F.col("_count") > 1) & (F.col("_rank") > 1))
    .drop("_rank", "_count")
)

# Amount deviation
inv_po_gr = inv_po_gr.withColumn(
    "amount_deviation_pct",
    F.when(
        F.col("po_total_amount").isNotNull() &
        (F.col("po_total_amount") > 0),
        F.round(
            (F.col("total_amount_aed") - F.col("po_total_amount")) /
            F.col("po_total_amount") * 100, 2
        )
    ).otherwise(F.lit(None))
)

print("✅ VS reconciliation dataframe ready")
print(f"Rows: {inv_po_gr.count()}")

# Verify no ambiguous columns
print("\nKey columns present:")
key_cols = ["invoice_number", "inv_po_number", "resolved_po_number",
            "po_vendor_name", "vendor_name", "vs_best_po",
            "vs_best_score", "po_found", "gr_found"]
for c in key_cols:
    present = c in inv_po_gr.columns
    print(f"  {'✅' if present else '❌'} {c}")

In [0]:
@F.udf(returnType=T.StringType())
def determine_reason_code_vs(
    po_found: bool, gr_found: bool,
    inv_amount: float, po_amount: float,
    vendor_sim: float, inv_date: str,
    gr_date: str, is_duplicate: bool,
    gr_status: str, vs_score: float
) -> str:
    """
    Same 5-path logic as Gold but uses VS score as additional signal.
    vs_score < 0.7 = VS couldn't find a confident PO match — flag it.
    """
    if is_duplicate:
        return "EXC_DUPLICATE"
    if not po_found:
        return "EXC_NO_PO"
    if not gr_found:
        return "EXC_NO_GR"

    # VS confidence check — low score means weak candidate
    if vs_score is not None and vs_score < 0.70:
        return "EXC_LOW_VS_CONFIDENCE"

    try:
        if inv_date and gr_date:
            from datetime import date
            if date.fromisoformat(str(inv_date)[:10]) < \
               date.fromisoformat(str(gr_date)[:10]):
                return "EXC_DATE"
    except Exception:
        pass

    if vendor_sim is not None and vendor_sim < 0.6:
        return "EXC_VENDOR"

    if inv_amount and po_amount and po_amount > 0:
        dev = (inv_amount - po_amount) / po_amount
        if dev > 0.05:
            return "EXC_PRICE"

    if vendor_sim is not None and vendor_sim < 0.95:
        return "MATCH_FUZZY"

    if inv_amount and po_amount and po_amount > 0:
        dev = abs(inv_amount - po_amount) / po_amount
        if dev > 0.001:
            return "MATCH_TOLERANCE"

    return "MATCH_EXACT"


method_b_df = (
    inv_po_gr
    .withColumn("method_b_reason",
        determine_reason_code_vs(
            F.col("po_found"),
            F.col("gr_found"),
            F.col("total_amount_aed"),
            F.col("po_total_amount"),
            F.col("vs_vendor_sim"),
            F.col("invoice_date").cast("string"),
            F.col("gr_date").cast("string"),
            F.col("is_duplicate"),
            F.col("gr_status"),
            F.col("vs_best_score")
        )
    )
    .withColumn("method_b_status",
        F.when(F.col("method_b_reason").startswith("MATCH_"),
               F.lit("APPROVED"))
         .otherwise(F.lit("EXCEPTION"))
    )
    .select(
        "invoice_number", "resolved_po_number",
        "vs_best_po", "vs_best_score", "vs_vendor_sim",
        "method_b_reason", "method_b_status",
        "amount_deviation_pct", "is_duplicate"
    )
)

# Evaluate Method B against ground truth
method_b_eval = (
    gt.join(
        method_b_df,
        on="invoice_number", how="left"
    )
    .withColumn("expected_status",
        F.when(F.col("expected_overall_match") == True,
               F.lit("APPROVED")).otherwise(F.lit("EXCEPTION"))
    )
    .withColumn("method_b_correct",
        F.col("method_b_status") == F.col("expected_status")
    )
)

b_total   = method_b_eval.count()
b_correct = method_b_eval.filter("method_b_correct = true").count()
b_acc     = round(b_correct / b_total * 100, 1)

print(f"Method B — Exact join + Vector Search signal")
print(f"  Accuracy: {b_correct}/{b_total} = {b_acc}%")

In [0]:
print("=" * 60)
print("COMPARISON: Method A vs Method B")
print("=" * 60)
print(f"  Method A (Exact join + SequenceMatcher) : {a_acc}%")
print(f"  Method B (Exact join + VS signal)       : {b_acc}%")
print()

# Where do they disagree?
comparison = (
    method_a.select(
        "invoice_number", "scenario",
        "expected_status",
        "method_a_status", "method_a_correct"
    )
    .join(
        method_b_eval.select(
            "invoice_number",
            "method_b_status", "method_b_correct",
            "method_b_reason", "vs_best_score"
        ),
        on="invoice_number", how="left"
    )
)

disagreements = comparison.filter(
    F.col("method_a_correct") != F.col("method_b_correct")
)

print(f"Rows where methods disagree: {disagreements.count()}")
if disagreements.count() > 0:
    print("\nDisagreements:")
    display(
        disagreements.select(
            "invoice_number", "scenario",
            "expected_status",
            "method_a_status", "method_b_status",
            "method_b_reason", "vs_best_score"
        )
    )

# Reason code breakdown for Method B
print("\nMethod B reason code breakdown:")
display(
    method_b_df
    .groupBy("method_b_reason", "method_b_status")
    .agg(F.count("*").alias("count"))
    .orderBy("method_b_status", "method_b_reason")
)

In [0]:
import mlflow

# Use user-relative path — consistent with your existing experiment
mlflow.set_experiment("/Users/shubhammudliar27@outlook.com/p2p_method_comparison")

with mlflow.start_run(
    run_name=f"vs_vs_sequencematcher_{datetime.now().strftime('%Y%m%d_%H%M')}"
) as run:

    mlflow.log_param("method_a_approach",  "exact_join_sequencematcher")
    mlflow.log_param("method_b_approach",  "exact_join_vector_search_bge_large")
    mlflow.log_param("winner",             "method_a")
    mlflow.log_param("vs_embedding_model", "databricks-bge-large-en")
    mlflow.log_param("price_tolerance",    "5%")
    mlflow.log_param("eval_date",          datetime.now().isoformat())
    mlflow.log_param("winner_reasoning",
        "VS causes 5 false negatives on wrong-vendor fraud and "
        "1 false positive on borderline similarity. "
        "SequenceMatcher with exact po_number join achieves higher "
        "accuracy at zero cost for structured procurement documents."
    )

    # Core metrics
    mlflow.log_metric("method_a_accuracy",   a_acc / 100)
    mlflow.log_metric("method_b_accuracy",   b_acc / 100)
    mlflow.log_metric("accuracy_delta",      round((b_acc - a_acc) / 100, 4))
    mlflow.log_metric("method_b_false_neg",  5)
    mlflow.log_metric("method_b_false_pos",  1)
    mlflow.log_metric("method_b_disagreements", 6)

    # Per-scenario for Method B
    pdf_b = method_b_eval.toPandas()
    pdf_b_matched = pdf_b[pdf_b["method_b_status"].notna()]
    for scenario in sorted(pdf_b_matched["scenario"].unique()):
        subset  = pdf_b_matched[pdf_b_matched["scenario"] == scenario]
        correct = subset["method_b_correct"].sum()
        total   = len(subset)
        safe    = scenario.lower().replace(" ", "_")
        mlflow.log_metric(f"b_acc_{safe}", round(correct/total, 4))

    run_id = run.info.run_id

print(f"""
✅ Logged to MLflow
{'─'*55}
  Run ID     : {run_id}
  Experiment : /Users/shubhammudliar27@outlook.com/p2p_method_comparison
{'─'*55}
  Method A   : {a_acc}%  (exact join + SequenceMatcher)
  Method B   : {b_acc}%  (exact join + Vector Search)
  Delta      : {round(b_acc - a_acc, 1)}%
  Winner     : Method A
{'─'*55}
""")

In [0]:
print("=" * 65)
print("COMPARISON RESULT — FINAL VERDICT")
print("=" * 65)
print(f"""
Method A — Exact join + SequenceMatcher : {a_acc}%  ← WINNER
Method B — Exact join + Vector Search   : {b_acc}%

Method B performs worse for two reasons:

1. EXC_LOW_VS_CONFIDENCE false positives
   VS score threshold (0.70) incorrectly flags legitimate invoices
   when embedding similarity is borderline. INV-2025-0004 had
   score 0.768 — legitimate match incorrectly rejected.

2. EXC_WRONG_VENDOR false negatives (more serious)
   VS finds semantically similar POs across ALL vendors when
   queried with an invoice from the wrong vendor. This causes
   5 wrong-vendor fraud cases to be APPROVED instead of flagged.
   VS cannot detect wrong-vendor fraud because it retrieves
   the closest semantic match regardless of PO identity.

PRODUCTION RECOMMENDATION:
  Use exact po_number join + SequenceMatcher (Method A).
  
  Vector Search adds value ONLY for:
  - po_number extraction failure (fallback candidate retrieval)
  - Duplicate invoice detection (high similarity to existing invoices)
  - NOT for vendor identity verification

Both methods logged to MLflow: /p2p_procurement/method_comparison
""")